# Deckard DVCLive: Native Runtime Walkthrough

This notebook demonstrates Deckard-native DVCLive behavior using a real default example experiment from `examples/sklearn/config/default.yaml`.

No mock experiment objects are used in this flow.

In [1]:
from __future__ import annotations

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from deckard.artifacts import ScoreDict
from deckard.experiment.dvc import run_dvc_experiment_plugin_hook


def ensure_dvclive_available() -> None:
    if importlib.util.find_spec("dvclive") is not None:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dvclive"])


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "deckard").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing pyproject.toml")


ensure_dvclive_available()

REPO_ROOT = find_repo_root(Path.cwd())
DVCLIVE_DIR = REPO_ROOT / "docs/notebooks/build/dvclive_native"
SCORE_ROOT = REPO_ROOT / "examples/sklearn/outputs/logs"

DVCLIVE_DIR

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PosixPath('/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive_native')

## Build A Real Default Experiment

Compose the default sklearn experiment config and instantiate a real `ExperimentConfig` object.

In [2]:
from deckard.experiment.base import ExperimentConfig


def build_default_example_experiment(repo_root: Path):
    config_dir = repo_root / "examples/sklearn/config"
    GlobalHydra.instance().clear()
    with initialize_config_dir(version_base=None, config_dir=config_dir.as_posix()):
        cfg = compose(config_name="default")

    payload = OmegaConf.to_container(cfg, resolve=True)
    if not isinstance(payload, dict):
        raise TypeError("Expected dict-like Hydra payload for experiment config")

    allowed_keys = set(ExperimentConfig.__dataclass_fields__.keys()) | {"_target_"}
    filtered = {key: value for key, value in payload.items() if key in allowed_keys}
    filtered.setdefault("_target_", "deckard.ExperimentConfig")

    experiment = instantiate(filtered)
    experiment.experiment_name = f"{getattr(experiment, 'experiment_name', 'default')}-dvclive-native"
    return experiment, cfg


experiment, default_cfg = build_default_example_experiment(REPO_ROOT)
type(experiment).__name__, experiment.experiment_name

('ExperimentConfig', 'f117485ab41a0c260f323aaa69e60d61-dvclive-native')

## Configure Native DVCLive Plugin Settings

Enable Deckard DVC hooks and point outputs to a notebook-local build directory.

In [3]:
if DVCLIVE_DIR.exists():
    shutil.rmtree(DVCLIVE_DIR)
DVCLIVE_DIR.mkdir(parents=True, exist_ok=True)

plugin_cfg = {
    "enabled": True,
    "dvclive_dir": DVCLIVE_DIR.as_posix(),
    "mode": "single",
    "pull_dependencies": False,
    "push_outputs": False,
    "make_summary": True,
    "make_report": True,
    "make_dvcyaml": False,
    "report_mode": "html",
    "resume": False,
    "save_dvc_exp": False,
    "cache_images": False,
    "monitor_system": True,
    "fail_on_dvc_error": False,
}

experiment.dvc_plugin = plugin_cfg
experiment.score_dict = ScoreDict.from_payload({})
plugin_cfg

{'enabled': True,
 'dvclive_dir': '/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive_native',
 'mode': 'single',
 'pull_dependencies': False,
 'push_outputs': False,
 'make_summary': True,
 'make_report': True,
 'make_dvcyaml': False,
 'report_mode': 'html',
 'resume': False,
 'save_dvc_exp': False,
 'cache_images': False,
 'monitor_system': True,
 'fail_on_dvc_error': False}

## Run Real Stage Scoring And DVC Hooks

Use real persisted example score files, then invoke Deckard's `run_dvc_experiment_plugin_hook` for load, score, and persist phases.

In [4]:
def find_score_file(tag: str) -> Path:
    candidates = sorted(SCORE_ROOT.glob("*/scores.json"))
    tagged = [path for path in candidates if tag in path.parent.name.lower()]
    if tagged:
        return tagged[0]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        f"No score files found under {SCORE_ROOT}. Run the sklearn example first."
    )


def load_numeric_scores(path: Path) -> dict[str, float]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        return {}
    if isinstance(payload.get("flat"), dict):
        source = payload["flat"]
    elif isinstance(payload.get("test"), dict):
        source = payload["test"]
    else:
        source = payload
    return {
        str(key): float(value)
        for key, value in source.items()
        if isinstance(value, (int, float))
    }


score_files = {
    "data-score": find_score_file("nomodel"),
    "model-score": find_score_file("model"),
    "attack-score": find_score_file("attack"),
}

hook_results = []
hook_results.append(
    run_dvc_experiment_plugin_hook(
        experiment,
        dvc_plugin=plugin_cfg,
        plugin_position="first",
        component="experiment",
        stage="load",
        event="before",
    )
)

for stage_name, score_path in score_files.items():
    numeric_scores = load_numeric_scores(score_path)
    experiment.score_dict.update_score(numeric_scores, stage=stage_name, mode="test")

    component = stage_name.split("-", 1)[0]
    hook_results.append(
        run_dvc_experiment_plugin_hook(
            experiment,
            dvc_plugin=plugin_cfg,
            plugin_position="last",
            component=component,
            stage=stage_name,
            event="after",
        )
    )

hook_results.append(
    run_dvc_experiment_plugin_hook(
        experiment,
        dvc_plugin=plugin_cfg,
        plugin_position="last",
        component="experiment",
        stage="score",
        event="after",
    )
)

persist_result = run_dvc_experiment_plugin_hook(
    experiment,
    dvc_plugin=plugin_cfg,
    plugin_position="last",
    component="experiment",
    stage="persist",
    event="after",
)

{
    "score_files": {k: v.as_posix() for k, v in score_files.items()},
    "hook_events": len(hook_results),
    "persist_result": persist_result,
}

{'score_files': {'data-score': '/Users/c.meyers/Documents/deckard/examples/sklearn/outputs/logs/anjana_chain_nomodel_006bb62e/scores.json',
  'model-score': '/Users/c.meyers/Documents/deckard/examples/sklearn/outputs/logs/anjana_chain_model_02940ea1/scores.json',
  'attack-score': '/Users/c.meyers/Documents/deckard/examples/sklearn/outputs/logs/anjana_chain_attack_05b3a956/scores.json'},
 'hook_events': 5,
 'persist_result': {'enabled': True,
  'position': 'last',
  'component': 'experiment',
  'stage': 'persist',
  'event': 'after',
  'executed': True,
  'dvclive_dir': 'docs/notebooks/build/dvclive_native',
  'report_mode': 'html',
  'summary_json': None,
  'report_file': 'docs/notebooks/build/dvclive_native/report.html',
  'report_html': 'docs/notebooks/build/dvclive_native/report.html',
  'system_monitor_scores': {'system_monitor/cpu/count': 12.0,
   'system_monitor/cpu/usage (%)': 0.0,
   'system_monitor/cpu/parallelization (%)': 0.0,
   'system_monitor/ram/usage (%)': 67.6,
   'sy

## Inspect Generated DVCLive Artifacts

Check generated summary/report files and confirm system-monitor fields merged into Deckard score storage.

In [5]:
summary_path = DVCLIVE_DIR / "summary.json"
report_path = DVCLIVE_DIR / "report.html"

summary_payload = {}
if summary_path.exists():
    summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))

system_monitor_keys = sorted(
    [key for key in experiment.score_dict.keys() if str(key).startswith("system_monitor/")]
)

{
    "dvclive_dir": DVCLIVE_DIR.as_posix(),
    "summary_exists": summary_path.exists(),
    "report_exists": report_path.exists(),
    "summary_top_level_keys": sorted(summary_payload.keys()) if summary_payload else [],
    "system_monitor_keys": system_monitor_keys,
    "system_monitor_count": len(system_monitor_keys),
}

{'dvclive_dir': '/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive_native',
 'summary_exists': False,
 'report_exists': True,
 'summary_top_level_keys': [],
 'system_monitor_keys': ['system_monitor/cpu/count',
  'system_monitor/cpu/parallelization (%)',
  'system_monitor/cpu/usage (%)',
  'system_monitor/disk/total (GB)/main',
  'system_monitor/disk/usage (%)/main',
  'system_monitor/disk/usage (GB)/main',
  'system_monitor/ram/total (GB)',
  'system_monitor/ram/usage (%)',
  'system_monitor/ram/usage (GB)'],
 'system_monitor_count': 9}